In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

/home/lol/.local/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.9.0' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/lol/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
/home/lol/.local/lib/python3.12/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

In [3]:
df = pd.read_csv("../data/processed/dataset_clean.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (121869, 34)
  cell_id   lat  lon        date  year  doy  label  temp_max  temp_min  \
0  340_60  34.0  6.0  2013-06-27  2013  178      0      34.4      22.4   
1  340_60  34.0  6.0  2013-07-04  2013  185      0      38.5      27.7   
2  340_60  34.0  6.0  2013-07-15  2013  196      0      38.5      27.4   
3  340_60  34.0  6.0  2013-09-24  2013  267      0      30.9      20.9   
4  340_60  34.0  6.0  2014-08-21  2014  233      1      40.1      28.5   

   temp_mean  ...        isi         bui        fwi  days_since_last_fire  \
0       28.5  ...  34.680699  515.655797  85.693776                9999.0   
1       33.8  ...  24.668329  565.143193  70.063644                9999.0   
2       33.5  ...  39.857688  645.171948  92.909470                9999.0   
3       26.3  ...  10.824992  145.202146  38.343835                9999.0   
4       35.1  ...  19.509543  666.450994  60.802735                9999.0   

   split      ndvi  ndvi_normal  ndvi_anomaly  ndvi_change_32d  

In [4]:
features = [
    'doy',
    'temp_max',
    'temp_min',
    'temp_mean',
    'rh_min',
    'rh_mean',
    'wind_max',
    'wind_mean',
    'precip_sum',
    'vpd_max',
    'precip_7d',
    'precip_30d',
    'days_since_rain_1mm',
    'temp_max_7d_mean',
    'rh_min_7d_mean',
    'ffmc',
    'dmc',
    'dc',
    'isi',
    'bui',
    'fwi',
    'ndvi',
    'ndvi_normal',
    'ndvi_change_32d',
    'ndvi_stale_days'
]

train_df = df[df['split'] == 'train']
val_df = df[df['split'] == 'val']
test_df = df[df['split'] == 'test']

x_train = train_df[features]
y_train = train_df['label']

x_val = val_df[features]
y_val = val_df['label']

x_test = test_df[features]
y_test = test_df['label']

print("Train:", x_train.shape)
print("Validation:", x_val.shape)
print("Test:", x_test.shape)

Train: (104979, 25)
Validation: (8423, 25)
Test: (8467, 25)


In [5]:
imputer = SimpleImputer(strategy='median')

x_train_imputed = imputer.fit_transform(x_train)
x_val_imputed = imputer.transform(x_val)
x_test_imputed = imputer.transform(x_test)

print("Train NaNs:", np.isnan(x_train_imputed).sum())
print("Validation NaNs:", np.isnan(x_val_imputed).sum())
print("Test NaNs:", np.isnan(x_test_imputed).sum())

Train NaNs: 0
Validation NaNs: 0
Test NaNs: 0


In [6]:
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train_imputed)
x_val_scaled = scaler.transform(x_val_imputed)
x_test_scaled = scaler.transform(x_test_imputed)

print("Train shape:", x_train_scaled.shape)
print("Validation shape:", x_val_scaled.shape)

Train shape: (104979, 25)
Validation shape: (8423, 25)


In [7]:
pca = PCA(n_components=0.95)

x_train_pca = pca.fit_transform(x_train_scaled)
x_val_pca = pca.transform(x_val_scaled)
x_test_pca = pca.transform(x_test_scaled)

print("Original features:", x_train_scaled.shape[1])
print("Principal components:", x_train_pca.shape[1])
print("Explained variance:", round(pca.explained_variance_ratio_.sum(), 4))

Original features: 25
Principal components: 12
Explained variance: 0.9539


In [8]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import average_precision_score

knn_pca = KNeighborsClassifier(
    n_neighbors=2001,
    metric='euclidean',
    weights='uniform',
    n_jobs=-1
)

knn_pca.fit(x_train_pca, y_train)

y_val_proba_pca = knn_pca.predict_proba(x_val_pca)[:, 1]

pr_auc_pca = average_precision_score(y_val, y_val_proba_pca)

print("PCA KNN Validation PR-AUC:", round(pr_auc_pca, 4))

PCA KNN Validation PR-AUC: 0.1052


In [9]:
pca_90 = PCA(n_components=0.90)

x_train_pca_90 = pca_90.fit_transform(x_train_scaled)
x_val_pca_90 = pca_90.transform(x_val_scaled)

print("Components:", x_train_pca_90.shape[1])
print("Explained variance:", round(pca_90.explained_variance_ratio_.sum(), 4))

Components: 9
Explained variance: 0.9093


In [10]:
knn_pca_90 = KNeighborsClassifier(
    n_neighbors=2001,
    metric='euclidean',
    weights='uniform',
    n_jobs=-1
)

knn_pca_90.fit(x_train_pca_90, y_train)

y_val_proba_90 = knn_pca_90.predict_proba(x_val_pca_90)[:, 1]

score_90 = average_precision_score(y_val, y_val_proba_90)

print("PCA 90% PR-AUC:", round(score_90, 4))

PCA 90% PR-AUC: 0.1049


In [11]:
pca_97 = PCA(n_components=0.97)

x_train_pca_97 = pca_97.fit_transform(x_train_scaled)
x_val_pca_97 = pca_97.transform(x_val_scaled)

print("Components:", x_train_pca_97.shape[1])
print("Explained variance:", round(pca_97.explained_variance_ratio_.sum(), 4))

Components: 14
Explained variance: 0.9738


In [12]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(knn_pca, "../models/knn_pca_95.joblib")
joblib.dump(pca, "../models/pca_95.joblib")
joblib.dump(imputer, "../models/imputer.joblib")
joblib.dump(scaler, "../models/scaler.joblib")

print("Models saved successfully!")

Models saved successfully!
